In [1]:
import sys, os
root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

In [4]:
import math
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForMaskedLM,
                          DataCollatorForLanguageModeling,
                          TrainingArguments, Trainer, AutoModel)
import evaluate

from pathlib import Path

class MLMConfig():
    SCIBERT_MODEL: str = "allenai/scibert_scivocab_uncased"
    PUBMED_BERT_MODEL: str = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext" 
    
    CSV_PATH: str = '/mnt/Supermicro/data2/chembl_35/chembl_35_activity_new.csv'
    PICKLE_PATH: str = '/mnt/Supermicro/data2/chembl_35/chembl_passages.pkl'
    TEXT_COL: str = 'passage'
    
#     BASE_DIR: Path = Path(__file__).resolve().parent.parent
#     MODELS_OUTPUT: Path = BASE_DIR / 'models'

    GRAD_ACCUM: int = 2
    EPOCHES: int = 3
    BATCH_PER_DEVICE: int = 48
    WARMUP_RATIO: float = 0.05
    LR: float = 5e-5

    RANDOM_SEED: int = 42
    TOKEN_MAX_LENGTH: int = 256

config = MLMConfig()

In [ ]:
data = pd.read_csv(config.CSV_PATH)

In [3]:
len(data)

18286963

In [3]:
data[[config.TEXT_COL]].isna().sum()

passage    0
dtype: int64

In [15]:
len(data) // 2


9143481

In [ ]:
data = data.sample(n=300000)

In [5]:
val_frac = 0.02
df_train = data.sample(frac=1-val_frac, random_state=config.RANDOM_SEED)
df_val = data.drop(df_train.index)

In [6]:
ds = DatasetDict({
    "train": Dataset.from_pandas(df_train[[config.TEXT_COL]], preserve_index=False),
    "validation": Dataset.from_pandas(df_val[[config.TEXT_COL]], preserve_index=False),
})

In [7]:
from peft import LoraConfig, get_peft_model, TaskType

peft_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,  
    r=16,                           
    lora_alpha=32,                 
    lora_dropout=0.1,             
    bias="none",
    modules_to_save=["lm_head"])

In [8]:
tokenizer = AutoTokenizer.from_pretrained(config.SCIBERT_MODEL, use_fast=False)
base_model = AutoModelForMaskedLM.from_pretrained(config.SCIBERT_MODEL)

model = get_peft_model(base_model, peft_config)

In [9]:
def tokenize_fn(batch):
    return tokenizer(batch[config.TEXT_COL],
                     truncation=True,
                     max_length=config.TOKEN_MAX_LENGTH,
                     return_attention_mask=True)

In [10]:
tokenized = ds.map(tokenize_fn, batched=True, remove_columns=[config.TEXT_COL], num_proc=64)

Map (num_proc=64):   0%|          | 0/1960000 [00:00<?, ? examples/s]

Map (num_proc=64):   0%|          | 0/40000 [00:00<?, ? examples/s]

In [24]:
import joblib

joblib.dump(tokenized, '../data/ds_tokenized.gz')


['../data/ds_tokenized.gz']

In [ ]:
tokenized = joblib.load('ds_tokenized.gz')
tokenized

In [11]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)


In [12]:
# perplexity = evaluate.load("perplexity", module_type="metric")

In [13]:
training_args = TrainingArguments(
    output_dir='scibert_scivocab_chembl_passages_v1',
    overwrite_output_dir=True,
    do_train=True,
    eval_strategy="epoch",
    learning_rate=config.LR,
    lr_scheduler_type="linear",
    warmup_ratio=config.WARMUP_RATIO,
    per_device_train_batch_size=config.BATCH_PER_DEVICE,
    per_device_eval_batch_size=config.BATCH_PER_DEVICE,
    gradient_accumulation_steps=config.GRAD_ACCUM,
    num_train_epochs=config.EPOCHES,
    weight_decay=0.02,
    seed=config.RANDOM_SEED,
    fp16=True,
    push_to_hub=True,
    hub_model_id='bitshott/scibert_scivocab_chembl_passages_v1'
)

In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer
    )


/tmp/ipykernel_15489/483251921.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

/mnt/Supermicro/data2/EL-CAP/.venv/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss


/mnt/Supermicro/data2/EL-CAP/.venv/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/mnt/Supermicro/data2/EL-CAP/.venv/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/mnt/Supermicro/data2/EL-CAP/.venv/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/mnt/Supermicro/data2/EL-CAP/.venv/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/mnt

In [ ]:
trainer.push_to_hub()


In [18]:
import math

eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Perplexity: 1.82


In [26]:
import torch
torch.cuda.empty_cache()

In [45]:
tokenizer = AutoTokenizer.from_pretrained(config.SCIBERT_MODEL, use_fast=False)
ft_model = AutoModel.from_pretrained('bitshott/scibert_scivocab_chembl_passages_v1').to('cuda:0')

Loading adapter weights from bitshott/scibert_scivocab_chembl_passages_v1 led to unexpected keys not found in the model: bert.encoder.layer.0.attention.self.query.lora_A.default.weight, bert.encoder.layer.0.attention.self.query.lora_B.default.weight, bert.encoder.layer.0.attention.self.value.lora_A.default.weight, bert.encoder.layer.0.attention.self.value.lora_B.default.weight, bert.encoder.layer.1.attention.self.query.lora_A.default.weight, bert.encoder.layer.1.attention.self.query.lora_B.default.weight, bert.encoder.layer.1.attention.self.value.lora_A.default.weight, bert.encoder.layer.1.attention.self.value.lora_B.default.weight, bert.encoder.layer.10.attention.self.query.lora_A.default.weight, bert.encoder.layer.10.attention.self.query.lora_B.default.weight, bert.encoder.layer.10.attention.self.value.lora_A.default.weight, bert.encoder.layer.10.attention.self.value.lora_B.default.weight, bert.encoder.layer.11.attention.self.query.lora_A.default.weight, bert.encoder.layer.11.attenti

In [23]:
data.to_pickle('../data/chembl_data_subsample.pkl')

In [46]:
text = "Inhibitory concentration against human DNA topoisomerase II"
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to('cuda:0')
with torch.no_grad():
    outputs = ft_model(**inputs)
    hidden_states = outputs.last_hidden_state

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [57]:
attention_mask = inputs["attention_mask"]
mask_expanded = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
sum_embeddings = torch.sum(hidden_states * mask_expanded, 1)
sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
mean_emb = sum_embeddings / sum_mask

In [59]:
attention_mask

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')